# Clase 3 — Clasificación, regresión y la Functional API de Keras

Curso "Fundamentos de TensorFlow desde cero" — DIH DATAlife (Hub de Innovación Digital Europeo, EDIH), a partir de la certificación EITC/AI/TFF (European IT Certificate / Artificial Intelligence / TensorFlow Fundamentals) "TensorFlow Fundamentals".

## Objetivos de esta clase

- Diferenciar problemas de clasificación y de regresión y elegir la función de pérdida adecuada para cada uno
- Cargar y preparar datos tabulares en formato CSV (Comma-Separated Values, valores separados por comas) para entrenar un modelo
- Construir modelos con la Functional API de Keras para arquitecturas no secuenciales
- Reconocer las diferencias clave entre TensorFlow 1.x y TensorFlow 2.0

Este notebook acompaña al capítulo 05 del manual del curso. Ejecuta las celdas en orden: cada bloque de código está comentado para que entiendas qué hace y por qué.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd

print("Versión de TensorFlow:", tf.__version__)
print("Ejecución eager activada:", tf.executing_eagerly())

## 1. Clasificación frente a regresión

Dos preguntas distintas necesitan dos formas distintas de entrenar un modelo:

- **Clasificación**: la salida es una categoría de un conjunto cerrado ("¿qué prenda es?", "¿el tumor es benigno o maligno?")
- **Regresión**: la salida es un número continuo ("¿cuántos kilómetros por litro hace este coche?")

Esa diferencia cambia la última capa de la red y, sobre todo, la función de pérdida:

| Problema | Última capa | Pérdida |
|---|---|---|
| Clasificación binaria | 1 neurona, activación `sigmoid` | `binary_crossentropy` |
| Clasificación multiclase | *n* neuronas (logits o `softmax`) | `sparse_categorical_crossentropy` |
| Regresión | 1 neurona, sin activación (lineal) | `mean_squared_error` o `mean_absolute_error` |

### 1.1 Ejemplo de clasificación: Fashion MNIST

Vamos a entrenar un clasificador multiclase con el dataset Fashion MNIST (10 categorías de ropa, imágenes de 28x28 píxeles en escala de grises).

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

class_names = ['Camiseta', 'Pantalon', 'Jersey', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botin']

# Normalizamos los píxeles al rango 0-1 (estaban en 0-255)
train_images = train_images / 255.0
test_images = test_images / 255.0

print("Imágenes de entrenamiento:", train_images.shape)
print("Etiquetas de entrenamiento:", train_labels.shape)

In [ ]:
# Modelo Sequential: una pila simple de capas, suficiente para este problema
modelo_clasificacion = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),   # aplana la imagen 28x28 a un vector de 784
    tf.keras.layers.Dense(128, activation='relu'),   # capa oculta
    tf.keras.layers.Dense(10)                        # 10 salidas (logits), una por clase
])

# Clasificación multiclase -> sparse_categorical_crossentropy
# from_logits=True porque la última capa no tiene activación softmax
modelo_clasificacion.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

# Entrenamos pocas épocas para que la celda se ejecute rápido en clase
modelo_clasificacion.fit(train_images, train_labels, epochs=5, verbose=1)

In [ ]:
test_loss, test_acc = modelo_clasificacion.evaluate(test_images, test_labels, verbose=0)
print(f"Precisión sobre el conjunto de prueba: {test_acc:.2%}")

# Convertimos los logits en probabilidades con una capa Softmax para interpretar la salida
modelo_probabilidades = tf.keras.Sequential([modelo_clasificacion, tf.keras.layers.Softmax()])
prediccion = modelo_probabilidades.predict(test_images[:1], verbose=0)
print("Clase predicha:", class_names[np.argmax(prediccion[0])])
print("Clase real:", class_names[test_labels[0]])

### 1.2 Ejemplo de regresión: consumo de combustible (Auto MPG)

Ahora resolvemos un problema de regresión: predecir las millas por galón (MPG, *miles per gallon*, una medida de consumo) de coches de los años setenta a partir de sus características (cilindros, peso, caballos de fuerza, entre otras).

In [ ]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
nombres_columnas = ['MPG', 'Cilindros', 'Desplazamiento', 'Caballos', 'Peso',
                     'Aceleracion', 'Año', 'Origen']

dataset_crudo = pd.read_csv(url, names=nombres_columnas, na_values='?',
                             comment='\t', sep=' ', skipinitialspace=True)

dataset = dataset_crudo.dropna()  # eliminamos filas con valores desconocidos

# 'Origen' es categórica (1=EEUU, 2=Europa, 3=Japón): la convertimos a one-hot
dataset['Origen'] = dataset['Origen'].map({1: 'EEUU', 2: 'Europa', 3: 'Japon'})
dataset = pd.get_dummies(dataset, columns=['Origen'], prefix='', prefix_sep='')

dataset.tail()

In [ ]:
# División en entrenamiento (80%) y prueba (20%)
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

# Separamos la etiqueta (lo que queremos predecir) de las características
train_features = train_dataset.copy()
test_features = test_dataset.copy()
train_labels = train_features.pop('MPG')
test_labels = test_features.pop('MPG')

# Capa de normalización: se "adapta" a las estadísticas del set de entrenamiento
normalizador = tf.keras.layers.Normalization(axis=-1)
normalizador.adapt(np.array(train_features))

In [ ]:
# Modelo de regresión: última capa con una neurona y sin activación (salida lineal)
modelo_regresion = tf.keras.Sequential([
    normalizador,
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])

# Regresión -> mean_absolute_error (o mean_squared_error), nunca accuracy
modelo_regresion.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='mean_absolute_error')

historial = modelo_regresion.fit(
    train_features, train_labels,
    validation_split=0.2,
    verbose=0, epochs=100)

error_test = modelo_regresion.evaluate(test_features, test_labels, verbose=0)
print(f"Error absoluto medio en el conjunto de prueba: {error_test:.2f} MPG")

## 2. Cargar y preparar datos CSV

La mayoría de datasets tabulares reales mezclan columnas numéricas y categóricas, y a menudo tienen escalas muy distintas. Vamos a ver dos técnicas:

1. Normalizar columnas numéricas con `tf.keras.layers.Normalization`
2. Convertir columnas categóricas a one-hot con `StringLookup` + `CategoryEncoding`, usando el dataset del Titanic como ejemplo.

In [ ]:
titanic = pd.read_csv("https://storage.googleapis.com/tf-datasets/titanic/train.csv")
titanic_features = titanic.copy()
titanic_labels = titanic_features.pop('survived')  # 0 = no sobrevivió, 1 = sobrevivió

titanic.head()

In [ ]:
# Creamos una entrada simbólica (Input) por cada columna del CSV, respetando su tipo
inputs = {}
for nombre, columna in titanic_features.items():
    dtype = tf.string if columna.dtype == object else tf.float32
    inputs[nombre] = tf.keras.Input(shape=(1,), name=nombre, dtype=dtype)

# Las columnas numéricas se concatenan y se pasan por una capa de normalización
entradas_numericas = {nombre: entrada for nombre, entrada in inputs.items()
                       if entrada.dtype == tf.float32}
x = tf.keras.layers.Concatenate()(list(entradas_numericas.values()))
normalizador_titanic = tf.keras.layers.Normalization()
normalizador_titanic.adapt(np.array(titanic[entradas_numericas.keys()]))
numericas_procesadas = normalizador_titanic(x)

preprocesado = [numericas_procesadas]

# Las columnas de texto se convierten en índices (StringLookup) y luego en one-hot (CategoryEncoding)
for nombre, entrada in inputs.items():
    if entrada.dtype == tf.float32:
        continue
    lookup = tf.keras.layers.StringLookup(vocabulary=np.unique(titanic_features[nombre]))
    one_hot = tf.keras.layers.CategoryEncoding(num_tokens=lookup.vocabulary_size())
    x = one_hot(lookup(entrada))
    preprocesado.append(x)

# Este modelo solo hace preprocesamiento: no predice nada todavía
preprocesado_concatenado = tf.keras.layers.Concatenate()(preprocesado)
titanic_preprocesamiento = tf.keras.Model(inputs, preprocesado_concatenado)
print("Preprocesamiento definido con", len(inputs), "columnas de entrada")

Fíjate en algo importante: para definir este preprocesamiento hemos usado `tf.keras.Input` y hemos encadenado capas llamándolas como funciones sobre tensores (`normalizador_titanic(x)`, `lookup(entrada)`). Eso es exactamente la **Functional API**, que vemos en detalle en la siguiente sección: la necesitamos aquí porque el modelo tiene tantas entradas como columnas tenga el CSV, algo que la Sequential API no puede expresar.

In [ ]:
# Construimos el modelo completo: preprocesamiento + red densa de clasificación binaria
def construir_modelo_titanic(cabecera_preprocesamiento, entradas):
    cuerpo = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    entradas_procesadas = cabecera_preprocesamiento(entradas)
    resultado = cuerpo(entradas_procesadas)
    modelo = tf.keras.Model(entradas, resultado)
    modelo.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                    optimizer=tf.keras.optimizers.Adam())
    return modelo

modelo_titanic = construir_modelo_titanic(titanic_preprocesamiento, inputs)

# Los modelos con entradas nombradas esperan un diccionario de arrays, no una tabla de pandas
titanic_features_dict = {nombre: np.array(valor) for nombre, valor in titanic_features.items()}
modelo_titanic.fit(x=titanic_features_dict, y=titanic_labels, epochs=10, verbose=0)
print("Entrenamiento completado")

## 3. La Functional API de Keras

La Sequential API apila capas una detrás de otra: una entrada, un camino lineal, una salida. En cuanto necesitas **varias entradas**, **varias salidas** o conexiones que no siguen ese orden, cambias a la Functional API.

La idea clave: cada capa es una función que se aplica sobre un tensor.

```python
entrada = keras.Input(shape=(784,))
x = layers.Dense(64, activation="relu")(entrada)
x = layers.Dense(64, activation="relu")(x)
salida = layers.Dense(10)(x)
modelo = keras.Model(inputs=entrada, outputs=salida)
```

Vamos a construir un modelo con **dos entradas independientes** que se combinan antes de la salida final: un caso que la Sequential API no puede resolver.

In [ ]:
# Ejemplo: un modelo que combina datos numéricos y datos categóricos ya codificados
entrada_numerica = tf.keras.Input(shape=(8,), name="datos_numericos")
entrada_categorica = tf.keras.Input(shape=(4,), name="datos_categoricos")

# Cada entrada sigue su propio camino antes de combinarse
rama_numerica = tf.keras.layers.Dense(32, activation='relu')(entrada_numerica)
rama_numerica = tf.keras.layers.Dense(16, activation='relu')(rama_numerica)

rama_categorica = tf.keras.layers.Dense(16, activation='relu')(entrada_categorica)

# concatenate une las dos ramas en un único vector de características
combinado = tf.keras.layers.concatenate([rama_numerica, rama_categorica])
salida = tf.keras.layers.Dense(1, activation='sigmoid')(combinado)

modelo_multientrada = tf.keras.Model(
    inputs=[entrada_numerica, entrada_categorica],
    outputs=salida,
    name="modelo_dos_entradas")

modelo_multientrada.summary()

In [ ]:
# Datos de ejemplo (aleatorios) solo para comprobar que el modelo funciona de punta a punta
datos_numericos_ejemplo = np.random.random((100, 8)).astype("float32")
datos_categoricos_ejemplo = np.random.randint(2, size=(100, 4)).astype("float32")
etiquetas_ejemplo = np.random.randint(2, size=(100, 1)).astype("float32")

modelo_multientrada.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo_multientrada.fit(
    {"datos_numericos": datos_numericos_ejemplo, "datos_categoricos": datos_categoricos_ejemplo},
    etiquetas_ejemplo,
    epochs=3, verbose=1)

Con la Sequential API esto habría sido imposible: solo admite una entrada y un único camino. La Functional API, en cambio, te deja dibujar el modelo como un grafo: dos entradas, dos ramas, un punto de unión y una salida. La misma idea sirve para modelos con **varias salidas** (por ejemplo, predecir a la vez una prioridad y un departamento), con **capas compartidas** (la misma capa aplicada a dos entradas distintas) o con **conexiones residuales** (donde la salida de una capa se suma a la salida de una capa posterior, saltándose las capas intermedias).

## 4. TensorFlow 1.x frente a TensorFlow 2.0

El código que hemos escrito en todo este curso usa **ejecución eager**: cada operación se calcula al instante, como en Python normal. Así funcionaba TensorFlow **antes** de la versión 2.0, con sesiones y grafos:

```python
# Estilo TensorFlow 1.x (no lo ejecutes, solo para comparar)
import tensorflow as tf
a = tf.constant(2)
b = tf.constant(3)
c = tf.add(a, b)          # no calcula nada todavía, solo define el grafo

with tf.Session() as sess:
    resultado = sess.run(c)  # aquí sí se ejecuta el grafo
    print(resultado)
```

Compáralo con el mismo cálculo en TensorFlow 2.x:

In [ ]:
a = tf.constant(2)
b = tf.constant(3)
c = tf.add(a, b)
print(c)  # el resultado ya está disponible, sin sesión ni sess.run()

Cambios clave que trajo TensorFlow 2.0:

- **Ejecución eager por defecto**: sin `tf.Session` ni `tf.placeholder`, cada tensor tiene su valor al momento
- **Keras como API oficial de alto nivel**: antes convivían varias formas de construir modelos varias, ahora `tf.keras` es la recomendada
- **`tf.function`**: convierte una función Python en un grafo optimizado cuando necesitas más rendimiento, sin volver a sesiones
- **API más simple y consistente**, con menos código repetitivo
- **Un único formato de guardado** (`SavedModel`) compatible con TensorFlow Serving, TensorFlow Lite y TensorFlow.js

`tf.function` es útil, por ejemplo, en un bucle de entrenamiento personalizado con `tf.GradientTape` (la herramienta que calcula gradientes automáticamente):

In [ ]:
# Ejemplo de un paso de entrenamiento manual, típico de TensorFlow 2.x
optimizador = tf.keras.optimizers.Adam()
funcion_perdida = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

@tf.function  # convierte esta función en un grafo optimizado la primera vez que se llama
def paso_de_entrenamiento(modelo, imagenes, etiquetas):
    with tf.GradientTape() as tape:
        predicciones = modelo(imagenes, training=True)
        perdida = funcion_perdida(etiquetas, predicciones)
    gradientes = tape.gradient(perdida, modelo.trainable_variables)
    optimizador.apply_gradients(zip(gradientes, modelo.trainable_variables))
    return perdida

# Probamos el paso con un lote pequeño de Fashion MNIST
lote_imagenes = train_images[:32].astype("float32")
lote_etiquetas = train_labels[:32]
perdida_obtenida = paso_de_entrenamiento(modelo_clasificacion, lote_imagenes, lote_etiquetas)
print("Pérdida del lote de prueba:", float(perdida_obtenida))

## Resumen

- Clasificación: salida categórica, pérdidas tipo `*_crossentropy`, métrica `accuracy`
- Regresión: salida numérica continua, pérdidas `mean_squared_error` o `mean_absolute_error`
- Los datos CSV necesitan preprocesamiento: separar etiquetas, normalizar columnas numéricas, convertir categóricas a one-hot
- La Sequential API sirve para pilas simples de capas la Functional API es necesaria para varias entradas, varias salidas o conexiones no lineales
- TensorFlow 2.0 sustituyó sesiones y grafos por ejecución eager, con Keras como API oficial y `tf.function` disponible cuando hace falta rendimiento de grafo

Continúa con el notebook de práctica de esta clase: `clase03_practica_clasificacion_regresion_keras.ipynb`.